# Get Bioavailable Iron from USDA

In [17]:
# Parameters
SAVE_DFS = True

In [18]:
import ast
import os
import pandas as pd
import requests

from dotenv import load_dotenv
from IPython.display import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [19]:
# Load splits from CSVs
df_train = pd.read_csv('../data/train/df_train.csv')
df_val = pd.read_csv('../data/val/df_val.csv')
df_test = pd.read_csv('../data/test/df_test.csv')

df_train.head(2)

,image_url,camera_or_phone_prob,food_prob,dish_name,food_type,ingredients,portion_size,nutritional_profile,cooking_method,sub_dt,image_name
0,https://file.b18a.io/7832973280900104501_54585...,0.8,0.90,oysters,homemade food,['oysters'],{'oysters': '500g'},"{'fat_g': 5.0, 'protein_g': 20.0, 'calories_kc...",raw,20250710,7832973280900104501_545859_.jpeg
1,https://file.b18a.io/7835136777400102715_70587...,0.7,0.95,grilled steak,restaurant food,"['steak', 'broccoli', 'potato', 'tomato', 'sau...","{'steak': '250g', 'broccoli': '50g', 'potato':...","{'fat_g': 30.0, 'protein_g': 50.0, 'calories_k...",grilling,20250702,7835136777400102715_705873_.jpeg


In [20]:
# Combine dfs
df_all = pd.concat([df_train, df_val, df_test])
df_all = df_all.reset_index()

df_all.shape

(5000, 12)

In [21]:
df_all['dish_name']

0                              oysters
1                        grilled steak
2              sweet and sour potatoes
3                              hot pot
4                   stir-fried noodles
                     ...              
4995                       noodle soup
4996              braised chicken feet
4997                           hot pot
4998    vegetable and chicken sandwich
4999                mixed asian dishes
Name: dish_name, Length: 5000, dtype: object

In [22]:
df_all['portion_size'][0]

"{'oysters': '500g'}"

In [23]:
df_all['portion_size'] = df_all['portion_size'].apply(ast.literal_eval)

df_all['portion_size'][0]

{'oysters': '500g'}

In [24]:
# Get all ingredients
ingredients = set()
for row in df_all.itertuples():
    keys = row.portion_size.keys()
    ingredients.update(row.portion_size.keys())
print(f'{len(ingredients)=}')
ingredients

len(ingredients)=798


{'abalone',
 'almonds',
 'anchovies',
 'apple',
 'apple chips',
 'apples',
 'apricot',
 'apricots',
 'asparagus',
 'assorted desserts',
 'avocado',
 'baby corn',
 'bacon',
 'bagel',
 'baked beans',
 'baked goods',
 'baked potato',
 'baklava',
 'bamboo shoots',
 'banana',
 'bananas',
 'bao dough',
 'bar',
 'base',
 'basil',
 'batter',
 'bbq ribs',
 'bean sprouts',
 'beans',
 'beef',
 'beef liver',
 'beef patty',
 'beef ribs',
 'beef snack',
 'beef stew',
 'beef tripe',
 'beer',
 'beet',
 'beetroot',
 'beets',
 'bell pepper',
 'bell peppers',
 'berries',
 'beverage',
 'beverages',
 'bird',
 'birds',
 'biscuit',
 'biscuits',
 'bitter melon',
 'black beans',
 'black fungus',
 'black jelly',
 'black olive tapenade',
 'black pudding',
 'black rice',
 'black sapote',
 'blackberries',
 'blueberries',
 'boiled egg',
 'boiled eggs',
 'bok choy',
 'borscht',
 'bounty bar',
 'bread',
 'bread roll',
 'bread rolls',
 'breading',
 'brisket',
 'broad beans',
 'broccoli',
 'broth',
 'buckwheat',
 'bulg

In [25]:
nutrient_names = ['Iron, Fe', 'Calcium, Ca', 'Vitamin C, total ascorbic acid']

ingredients_df = {'ingredients': list(ingredients)}
ingredients_df = pd.DataFrame(ingredients_df)
for name in nutrient_names:
    ingredients_df[name] = ''
ingredients_df

,ingredients,"Iron, Fe","Calcium, Ca","Vitamin C, total ascorbic acid"
0,crabs,,,
1,beetroot,,,
2,coconut,,,
3,sea urchin,,,
4,pasta shells,,,
...,...,...,...,...
793,meat dish,,,
794,cantaloupe,,,
795,crust,,,
796,pork floss,,,


In [26]:
csv_file = '../data/ingredients.csv'

if SAVE_DFS:
    # Save to CSV
    ingredients_df.to_csv(csv_file, index=False)

# Load data from previously saved CSV
ingredients_df = pd.read_csv(csv_file)
ingredients_df

,ingredients,"Iron, Fe","Calcium, Ca","Vitamin C, total ascorbic acid"
0,crabs,NaN,NaN,NaN
1,beetroot,NaN,NaN,NaN
2,coconut,NaN,NaN,NaN
3,sea urchin,NaN,NaN,NaN
4,pasta shells,NaN,NaN,NaN
...,...,...,...,...
793,meat dish,NaN,NaN,NaN
794,cantaloupe,NaN,NaN,NaN
795,crust,NaN,NaN,NaN
796,pork floss,NaN,NaN,NaN


## USDA FoodData Central API

In [27]:
# Load and get api key from .env
load_dotenv()
api_key = os.getenv("API_KEY")

FOODS_SEARCH_URL = 'https://api.nal.usda.gov/fdc/v1/foods/search'

In [28]:
# Search for food
ingredient = 'abalone'
food_query = ingredient
r = requests.get(f'https://api.nal.usda.gov/fdc/v1/foods/search?api_key={api_key}&query={food_query}&dataType=Foundation')
data = r.json()
data_df = pd.DataFrame.from_dict(data['foods'])
if len(data_df) == 0:  # Data not found
    r = requests.get(f'https://api.nal.usda.gov/fdc/v1/foods/search?api_key={api_key}&query={food_query}')
    data = r.json()
    data_df = pd.DataFrame.from_dict(data['foods'])
data_df

,fdcId,description,commonNames,additionalDescriptions,dataType,foodCode,publishedDate,foodCategory,foodCategoryId,allHighlightFields,...,marketCountry,modifiedDate,dataSource,servingSizeUnit,servingSize,householdServingFullText,tradeChannels,brandName,packageWeight,shortDescription
0,2706337,Abalone,,,Survey (FNDDS),26301110.0,2024-10-31,Shellfish,3300222.0,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,174212,"Mollusks, abalone, mixed species, raw",,,SR Legacy,NaN,2019-04-01,Finfish and Shellfish Products,NaN,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,174213,"Mollusks, abalone, mixed species, cooked, fried",,,SR Legacy,NaN,2019-04-01,Finfish and Shellfish Products,NaN,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,552097,RICE PORRIDGE WITH ABALONE,NaN,NaN,Branded,NaN,2019-04-01,Other Soups,NaN,"<b>Ingredients</b>: WATER, RICE, <em>ABALONE</...",...,United States,2017-11-09,LI,g,285.0,10.05 ONZ,[NO_TRADE_CHANNEL],NaN,NaN,NaN
4,2085344,"NO. 1, WHOLE ABALONE MUSHROOM",NaN,NaN,Branded,NaN,2021-10-28,Canned Vegetables,NaN,<b>Ingredients</b>: <em>ABALONE</em> MUSHROOM ...,...,United States,2017-06-25,LI,g,100.0,13 PEASE,[NO_TRADE_CHANNEL],NO. 1,15 oz/425 g,NaN
5,2014577,"CALIFORNIA GIRL, IMITATION ABALONE",NaN,NaN,Branded,NaN,2021-10-28,Canned Seafood,NaN,,...,United States,2018-07-10,LI,g,55.0,0.5 cup,[NO_TRADE_CHANNEL],CALIFORNIA GIRL,15 oz/425 g,NaN
6,2176844,"WILD CAUGHT IMITATION ABALONE, WILD CAUGHT",NaN,NaN,Branded,NaN,2021-10-28,Canned Seafood,NaN,"<b>Ingredients</b>: CALAMARI, <em>ABALONE</em>...",...,United States,2021-04-23,LI,g,53.0,1/4 cup,[NO_TRADE_CHANNEL],SOL-MEX,15 OZ/425 G,NaN
7,2182958,Moana Select Harvest New Zealand Blue Abalone ...,NaN,NaN,Branded,NaN,2021-10-28,Seafood Miscellaneous,NaN,,...,New Zealand,2021-06-18,NZGDSN,g,45.0,None,[NO_TRADE_CHANNEL],Moana,200g,NaN
8,2070186,"WOODSTOCK, ORGANIC MIXED MUSHROOMS",NaN,NaN,Branded,NaN,2021-10-28,Frozen Vegetables,NaN,<b>Ingredients</b>: ORGANIC SHIITAKE MUSHROOMS...,...,United States,2017-07-14,LI,g,100.0,0.5 cup,[NO_TRADE_CHANNEL],WOODSTOCK,10 oz/283 g,NaN
9,2388182,"STIR FRY VEGETABLES, STIR FRY",NaN,NaN,Branded,NaN,2022-12-22,Canned Vegetables,NaN,"<b>Ingredients</b>: BEAN SPROUTS, BAMBOO SHOOT...",...,United States,2020-10-26,LI,g,230.0,1 can drained,[NO_TRADE_CHANNEL],ASIAN GOURMET,14 oz/397 G,


In [29]:
# Get food nutrients
nutrients = pd.DataFrame(data_df.iloc[0]['foodNutrients'])
print('Iron, Fe' in list(nutrients['nutrientName']))
nutrients = nutrients.loc[nutrients['nutrientName'].isin(nutrient_names)][['nutrientName', 'value']]
nutrients

True


,nutrientName,value
10,"Calcium, Ca",39.00
11,"Iron, Fe",3.97
28,"Vitamin C, total ascorbic acid",2.00


In [30]:
for row in nutrients.itertuples():
    print(row)
    ingredients_df.loc[ingredients_df['ingredients']==ingredient, row.nutrientName] = row.value
ingredients_df.loc[ingredients_df['ingredients']==ingredient]

Pandas(Index=10, nutrientName='Calcium, Ca', value=39.0)
Pandas(Index=11, nutrientName='Iron, Fe', value=3.97)
Pandas(Index=28, nutrientName='Vitamin C, total ascorbic acid', value=2.0)


,ingredients,"Iron, Fe","Calcium, Ca","Vitamin C, total ascorbic acid"
575,abalone,3.97,39.0,2.0


In [31]:
def get_nutrients(data_df, ingredients_df, nutrient_names):
    if len(data_df) > 0:  # Food found
        nutrients = pd.DataFrame()
        i = 0
        while len(nutrients) == 0 and i < len(data_df):  # Look for entry with nutrients
            nutrients = pd.DataFrame(data_df.iloc[i]['foodNutrients'])
            i += 1
            if len(nutrients) > 0:  # Nutrients found
                nutrients = nutrients.loc[nutrients['nutrientName'].isin(nutrient_names)][['nutrientName', 'value']]
                if len(nutrients) == 0:  # Skip if no desired nutrients
                    continue
                for row in nutrients.itertuples():
                    ingredients_df.loc[ingredients_df['ingredients']==ingredient, row.nutrientName] = row.value
                ingredients_df.loc[ingredients_df['ingredients']==ingredient, 'found'] = True
                break
    return ingredients_df

In [ ]:
# Fill CSV with nutrient data for ingredients
csv_file = '../data/ingredients_auto.csv'
ingredients_df['found'] = False

for ingredient in tqdm(ingredients):
    # Search for food
    food_query = ingredient
    r = requests.get(f'https://api.nal.usda.gov/fdc/v1/foods/search?api_key={api_key}&query={food_query}&dataType=Foundation')
    data = r.json()
    data_df = pd.DataFrame.from_dict(data['foods'])
    # Get food nutrients
    ingredients_df = get_nutrients(data_df, ingredients_df, nutrient_names)

    # Check if food not found earlier, broaden food search
    if ingredients_df.loc[ingredients_df['ingredients']==ingredient]['found'].item() == False:
        r = requests.get(f'https://api.nal.usda.gov/fdc/v1/foods/search?api_key={api_key}&query={food_query}')
        data = r.json()
        data_df = pd.DataFrame.from_dict(data['foods'])
        # Get food nutrients
        ingredients_df = get_nutrients(data_df, ingredients_df, nutrient_names)

    if SAVE_DFS:
        # Save to CSV
        ingredients_df.to_csv(csv_file, index=False)

# Load data from previously saved CSV
ingredients_df = pd.read_csv(csv_file)
ingredients_df

  1%|          | 9/798 [00:05<07:44,  1.70it/s]

In [ ]:
ingredients_df.loc[ingredients_df['found']==False]

,ingredients,"Iron, Fe","Calcium, Ca","Vitamin C, total ascorbic acid",found
8,drink1,NaN,NaN,NaN,False
31,condiment1,NaN,NaN,NaN,False
51,pastry4,NaN,NaN,NaN,False
75,efo riro,NaN,NaN,NaN,False
143,condiment2,NaN,NaN,NaN,False
189,drink2,NaN,NaN,NaN,False
223,pastry1,NaN,NaN,NaN,False
246,pastry2,NaN,NaN,NaN,False
306,snack3,NaN,NaN,NaN,False
359,nextar,NaN,NaN,NaN,False


In [ ]:
# Search for food
ingredient = 'porridge'
food_query = ingredient
r = requests.get(f'https://api.nal.usda.gov/fdc/v1/foods/search?api_key={api_key}&query={food_query}&dataType=Foundation')
data = r.json()
data_df = pd.DataFrame.from_dict(data['foods'])
if len(data_df) == 0:  # Data not found
    r = requests.get(f'https://api.nal.usda.gov/fdc/v1/foods/search?api_key={api_key}&query={food_query}')
    data = r.json()
    data_df = pd.DataFrame.from_dict(data['foods'])
data_df

,fdcId,description,dataType,publishedDate,tags,allHighlightFields,score,microbes,foodNutrients,finalFoodInputFoods,...,packageWeight,servingSizeUnit,servingSize,householdServingFullText,shortDescription,tradeChannels,commonNames,additionalDescriptions,foodCode,foodCategoryId
0,2687788,Potential of moringa leaf and baobab fruit foo...,Experimental,2024-04-18,"[Processing, Post-Harvest, USDA Research]",,303.968050,[],[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2417422,"MULTIGRAIN PORRIDGE SLICED BREAD, TARTINE MULT...",Branded,2022-12-22,NaN,,230.002460,[],"[{'nutrientId': 1003, 'nutrientName': 'Protein...",[],...,2 oz/500 g,g,50.0,1 slice,,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
2,2089884,ARTIFICIAL CHICKEN PORRIDGE,Branded,2021-10-28,NaN,<b>Ingredients</b>: <em>PORRIDGE</em> (52%): R...,202.842590,[],"[{'nutrientId': 1003, 'nutrientName': 'Protein...",[],...,4.2 oz/120 g,g,120.0,1 BOWL,NaN,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
3,2168938,ORGANIC PORRIDGE OATS,Branded,2021-10-28,NaN,,202.842590,[],"[{'nutrientId': 1003, 'nutrientName': 'Protein...",[],...,35 oz/2.19 LB/992 g,g,40.0,1/2 cup,NaN,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
4,1002697,PUMPKIN PORRIDGE WITH HONEY,Branded,2020-06-26,NaN,,202.842590,[],"[{'nutrientId': 1003, 'nutrientName': 'Protein...",[],...,NaN,g,285.0,10.05 ONZ,NaN,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
5,552097,RICE PORRIDGE WITH ABALONE,Branded,2019-04-01,NaN,,202.842590,[],"[{'nutrientId': 1004, 'nutrientName': 'Total l...",[],...,NaN,g,285.0,10.05 ONZ,NaN,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
6,552435,RICE PORRIDGE WITH TUNA,Branded,2019-04-01,NaN,,202.842590,[],"[{'nutrientId': 1087, 'nutrientName': 'Calcium...",[],...,NaN,g,285.0,10.05 ONZ,NaN,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
7,560546,RICE PORRIDGE WITH VEGETABLE,Branded,2019-04-01,NaN,,202.842590,[],"[{'nutrientId': 1004, 'nutrientName': 'Total l...",[],...,NaN,g,285.0,10.05 ONZ,NaN,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
8,2089623,SUPER SMOOTH PORRIDGE,Branded,2021-10-28,NaN,,202.842590,[],"[{'nutrientId': 1003, 'nutrientName': 'Protein...",[],...,NaN,g,100.0,100 GRM,NaN,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
9,2432400,"TEFF PORRIDGE, IVORY",Branded,2022-12-22,NaN,,202.842590,[],"[{'nutrientId': 1003, 'nutrientName': 'Protein...",[],...,5 oz/142 g,g,50.0,0.25 cup,,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
